<a href="https://colab.research.google.com/github/Soukthavilay/data-science-shoppee-thailand/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CÀI ĐẶT & TẢI DỮ LIỆU

In [5]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    log_loss, ConfusionMatrixDisplay
)

In [6]:
# Style chung cho biểu đồ
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#444466",
    "axes.labelcolor":  "#ccccee",
    "axes.titlecolor":  "#ffffff",
    "xtick.color":      "#aaaacc",
    "ytick.color":      "#aaaacc",
    "text.color":       "#ccccee",
    "grid.color":       "#2a2a4a",
    "grid.linewidth":   0.6,
    "font.family":      "monospace",
})
PALETTE = {"Completed": "#00d4aa", "Cancelled": "#ff6b8a", "Refunded": "#7b68ee"}

In [7]:
!pip install kagglehub -q

In [8]:
import kagglehub, os, glob

path = kagglehub.dataset_download(
    "hninshwezinhlaing/shopee-th-customer-journey-and-operations-dataset"
)
print("📁 Path to dataset:", path)

# Tìm tất cả file CSV trong dataset
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(f"📂 Tìm thấy {len(csv_files)} file CSV:")
for f in csv_files:
    print(f"   {os.path.basename(f)}")

100%|██████████| 70.9M/70.9M [00:00<00:00, 85.6MB/s]

Extracting files...


📁 Path to dataset: /root/.cache/kagglehub/datasets/hninshwezinhlaing/shopee-th-customer-journey-and-operations-dataset/versions/1
📂 Tìm thấy 11 file CSV:
   shopee_website_sessions_thailand.csv
   shopee_products_thailand.csv
   shopee_session_activities_thailand.csv
   shopee_product_campaign_thailand.csv
   shopee_sellers_thailand.csv
   shopee_campaigns_thailand.csv
   shopee_customers_thailand.csv
   shopee_reviews_thailand.csv
   shopee_order_items_thailand.csv
   shopee_orders_thailand.csv
   shopee_shipments_thailand.csv


# ĐỌC DỮ LIỆU VÀ CHUẨN HÓA DỮ LIỆU

In [9]:
BASE = "/root/.cache/kagglehub/datasets/hninshwezinhlaing/shopee-th-customer-journey-and-operations-dataset/versions/1"

# Đọc các file chính
customers  = pd.read_csv(f"{BASE}/shopee_customers_thailand.csv")
orders     = pd.read_csv(f"{BASE}/shopee_orders_thailand.csv")
order_items= pd.read_csv(f"{BASE}/shopee_order_items_thailand.csv")
products   = pd.read_csv(f"{BASE}/shopee_products_thailand.csv")
reviews    = pd.read_csv(f"{BASE}/shopee_reviews_thailand.csv")
sessions   = pd.read_csv(f"{BASE}/shopee_website_sessions_thailand.csv")

# Khám phá sơ bộ từng bảng
tables = {
    "customers":   customers,
    "orders":      orders,
    "order_items": order_items,
    "products":    products,
    "reviews":     reviews,
    "sessions":    sessions,
}

for name, df in tables.items():
    print(f"\n{'═'*55}")
    print(f" {name.upper()}  →  {df.shape[0]:,} dòng  x  {df.shape[1]} cột")
    print(f"{'═'*55}")
    print(df.dtypes.to_string())
    print(f"\n── 3 dòng đầu ──")
    print(df.head(3).to_string(index=False))


═══════════════════════════════════════════════════════
 CUSTOMERS  →  60,000 dòng  x  10 cột
═══════════════════════════════════════════════════════
customer_id          object
first_name           object
last_name            object
gender               object
dob                  object
registration_date    object
phone                 int64
email                object
province             object
city                 object

── 3 dòng đầu ──
customer_id first_name    last_name gender        dob registration_date     phone                               email     province         city
    C000001     Phokin Kanlayanamit   Male 1971-10-30        2022-01-01 799995582 phokin.kanlayanamit111447@gmail.com      Bangkok      Bangkok
    C000002       Anan Anantanakorn   Male 1983-09-11        2022-01-01 901952800   anan.anantanakorn444757@gmail.com Pathum Thani Pathum Thani
    C000003   Kritsada       Namsai   Male 1994-06-23        2022-01-01 903127521      kritsada.namsai31660@gmail.com  

Bài toán chính: Dự đoán item_status (Completed / Cancelled) — phù hợp cho Classification với Random Forest, dùng được trực tiếp từ order_items.

## Kiểm tra Missing Values & Phân phối nhãn

In [10]:
print("═" * 55)
print("  MISSING VALUES — order_items")
print("═" * 55)
missing = order_items.isnull().sum()
missing_pct = (missing / len(order_items) * 100).round(2)
summary = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
print(summary[summary["missing_count"] > 0].to_string())

═══════════════════════════════════════════════════════
  MISSING VALUES — order_items
═══════════════════════════════════════════════════════
                     missing_count  missing_%
product_campaign_id         470134      97.85


In [11]:
print("\n" + "═" * 55)
print("  PHÂN PHỐI NHÃN: item_status")
print("═" * 55)
label_dist = order_items["item_status"].value_counts()
label_pct  = order_items["item_status"].value_counts(normalize=True).mul(100).round(2)
print(pd.DataFrame({"count": label_dist, "%": label_pct}).to_string())


═══════════════════════════════════════════════════════
  PHÂN PHỐI NHÃN: item_status
═══════════════════════════════════════════════════════
              count      %
item_status               
Completed    360187  74.96
Cancelled     96229  20.03
Refunded      24065   5.01


In [12]:
print("\n" + "═" * 55)
print("  MISSING VALUES — products")
print("═" * 55)
missing_p = products.isnull().sum()
print(missing_p[missing_p > 0].to_string() or "  ✅ Không có missing values")


═══════════════════════════════════════════════════════
  MISSING VALUES — products
═══════════════════════════════════════════════════════
Series([], )


In [13]:
print("\n" + "═" * 55)
print("  PHÂN PHỐI CATEGORY (products)")
print("═" * 55)
print(products["category"].value_counts().to_string())


═══════════════════════════════════════════════════════
  PHÂN PHỐI CATEGORY (products)
═══════════════════════════════════════════════════════
category
Beauty         1085
Fashion        1078
Electronics     939
Groceries       931
Home            847


Mục tiêu: Biết nhãn có cân bằng không, missing values cần xử lý gì — trước khi chuẩn hóa và train

Kết quả rất rõ ràng! Nhãn hơi mất cân bằng (75/20/5) — sẽ xử lý bằng class_weight='balanced'. Tiếp theo build feature set và chuẩn hóa.

# Chuẩn hóa dữ liệu

In [14]:
# Merge order_items với products để lấy thêm category
df = order_items.merge(products[["product_id", "category", "commission_rate", "maintenance_rate"]],
                       on="product_id", how="left")

print(f"Sau merge: {df.shape[0]:,} dòng x {df.shape[1]} cột")

Sau merge: 480,481 dòng x 19 cột


In [15]:
NUMERIC_COLS  = [
    "quantity", "unit_price", "unit_price_after_discount",
    "discount_percent", "commission_amount", "maintenance_amount",
    "shipping_fee_item", "is_campaign", "commission_rate", "maintenance_rate",
]
le_cat    = LabelEncoder()
le_target = LabelEncoder()
df["category_enc"] = le_cat.fit_transform(df["category"])
df["label"]        = le_target.fit_transform(df["item_status"])
ALL_FEATURES = NUMERIC_COLS + ["category_enc"]

scaler = StandardScaler()
X = scaler.fit_transform(df[ALL_FEATURES])
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [20]:

import time

# ── Retrain KHÔNG dùng class_weight ──────────────────────────
params = dict(
    n_estimators      = 200,
    max_depth         = 10,
    min_samples_split = 5,
    random_state      = 42,
    n_jobs            = -1,
    # class_weight bỏ → để mô hình học phân phối thật
)
print("── Siêu tham số (đã bỏ class_weight) ──")
for k, v in params.items():
    print(f"   {k:<22}: {v}")

model2 = RandomForestClassifier(**params)
print(f"\n⏳ Đang huấn luyện lại...")
t0 = time.time()
model2.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"✅ Xong! ({elapsed:.1f} giây)")

── Siêu tham số (đã bỏ class_weight) ──
   n_estimators          : 200
   max_depth             : 10
   min_samples_split     : 5
   random_state          : 42
   n_jobs                : -1

⏳ Đang huấn luyện lại...
✅ Xong! (128.1 giây)


In [ ]:
# Chuẩn hóa numeric features (StandardScaler)
NUMERIC_COLS = [
    "quantity", "unit_price", "unit_price_after_discount",
    "discount_percent", "commission_amount", "maintenance_amount",
    "shipping_fee_item", "is_campaign",
    "commission_rate", "maintenance_rate", "category_enc"
]

scaler = StandardScaler()
X = scaler.fit_transform(df[NUMERIC_COLS])
y = df["label"].values

print(f"\n── Sau chuẩn hóa StandardScaler ──")
print(f"   Shape X : {X.shape}")
print(f"   Shape y : {y.shape}")
print(f"   mean X  : {X.mean(axis=0).round(4)}")
print(f"   std  X  : {X.std(axis=0).round(4)}")

In [ ]:
# Phân phối nhãn cuối
unique, counts = np.unique(y, return_counts=True)
print(f"\n── Phân phối nhãn sau encode ──")
for cls_idx, cnt in zip(unique, counts):
    print(f"   {le_target.classes_[cls_idx]:<12} ({cls_idx}) : {cnt:,}  ({cnt/len(y)*100:.1f}%)")

print("\n✅ Hoàn thành chuẩn hóa dữ liệu!")

## Vẽ 2 Biểu đồ

In [ ]:
fig1, ax1 = plt.subplots(figsize=(11, 6))
fig1.patch.set_facecolor("#0f0f1a")
ax1.set_facecolor("#1a1a2e")

pivot = (df.groupby(["category", "item_status"])
           .size()
           .unstack(fill_value=0))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

status_order = ["Completed", "Cancelled", "Refunded"]
colors = [PALETTE[s] for s in status_order]
pivot_pct[status_order].plot(
    kind="bar", stacked=True, ax=ax1,
    color=colors, edgecolor="#0f0f1a", linewidth=0.5, width=0.6
)

for container in ax1.containers:
    ax1.bar_label(container, fmt="%.1f%%", label_type="center",
                  fontsize=9, color="white", fontweight="bold")

ax1.set_title(
    "BIỂU ĐỒ 1 — Tỷ lệ Trạng thái Đơn hàng theo Danh mục sản phẩm\n"
    "(Stacked Bar Chart)",
    fontsize=13, fontweight="bold", pad=14, color="#ffffff"
)
ax1.set_xlabel("Danh mục sản phẩm", fontsize=11, labelpad=8)
ax1.set_ylabel("Tỷ lệ (%)", fontsize=11, labelpad=8)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0, fontsize=10)
ax1.legend(title="Trạng thái", title_fontsize=9,
           framealpha=0.15, labelcolor="white", fontsize=9)
ax1.grid(True, axis="y", alpha=0.25)
for sp in ax1.spines.values():
    sp.set_edgecolor("#333355")

plt.tight_layout()
plt.savefig("bieu_do_1_stacked_bar.png", dpi=150,
            bbox_inches="tight", facecolor="#0f0f1a")
plt.show()
print("✅ Đã lưu: bieu_do_1_stacked_bar.png")

In [ ]:
fig2, ax2 = plt.subplots(figsize=(11, 6))
fig2.patch.set_facecolor("#0f0f1a")
ax2.set_facecolor("#1a1a2e")

# Giới hạn trục X ở 1000 để loại bỏ outlier cực cao, nhìn rõ hơn
PRICE_CAP = 2000

for status, color in PALETTE.items():
    data = df[df["item_status"] == status]["unit_price_after_discount"]
    data = data[data <= PRICE_CAP]
    ax2.hist(data, bins=60, color=color, alpha=0.55,
             label=f"{status} (n={len(data):,})", edgecolor="none")

ax2.set_title(
    "BIỂU ĐỒ 2 — Phân phối Giá Sản phẩm (Unit Price) theo Trạng thái Đơn hàng\n"
    "(Histogram, giới hạn ≤ 2,000 THB)",
    fontsize=13, fontweight="bold", pad=14, color="#ffffff"
)
ax2.set_xlabel("Giá sau giảm (THB)", fontsize=11, labelpad=8)
ax2.set_ylabel("Số lượng sản phẩm", fontsize=11, labelpad=8)
ax2.legend(framealpha=0.15, labelcolor="white", fontsize=10)
ax2.grid(True, axis="y", alpha=0.25)
for sp in ax2.spines.values():
    sp.set_edgecolor("#333355")

plt.tight_layout()
plt.savefig("bieu_do_2_histogram.png", dpi=150,
            bbox_inches="tight", facecolor="#0f0f1a")
plt.show()
print("✅ Đã lưu: bieu_do_2_histogram.png")

3 nhóm đều tập trung ở giá thấp (< 300 THB)

# Train mô hình Random Forest

In [ ]:
# Chuẩn bị features (lặp lại từ Bước 3)
NUMERIC_COLS = [
    "quantity", "unit_price", "unit_price_after_discount",
    "discount_percent", "commission_amount", "maintenance_amount",
    "shipping_fee_item", "is_campaign",
    "commission_rate", "maintenance_rate",
]

In [ ]:
le_cat    = LabelEncoder()
le_target = LabelEncoder()
df["category_enc"] = le_cat.fit_transform(df["category"])
df["label"]        = le_target.fit_transform(df["item_status"])

In [ ]:
ALL_FEATURES = NUMERIC_COLS + ["category_enc"]

scaler  = StandardScaler()
X       = scaler.fit_transform(df[ALL_FEATURES])
y       = df["label"].values

In [ ]:
# --- Chia train / test (80/20, stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"── Chia dữ liệu ──")
print(f"   Train : {X_train.shape[0]:,} mẫu")
print(f"   Test  : {X_test.shape[0]:,} mẫu")

In [ ]:
# --- Định nghĩa mô hình
#     class_weight='balanced' xử lý mất cân bằng nhãn (75/20/5)
#     n_estimators=200 đủ ổn định, n_jobs=-1 dùng tất cả CPU core
params = dict(
    n_estimators   = 200,
    max_depth      = 10,
    min_samples_split = 5,
    class_weight   = "balanced",
    random_state   = 42,
    n_jobs         = -1,
)

print(f"\n── Siêu tham số ──")
for k, v in params.items():
    print(f"   {k:<22}: {v}")

In [ ]:
# --- Huấn luyện
import time

model = RandomForestClassifier(**params)
print(f"\n⏳ Đang huấn luyện...")
t0 = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"✅ Huấn luyện xong!  ({elapsed:.1f} giây)")

In [ ]:
# --- Feature Importance
importances = model.feature_importances_
feat_names  = ALL_FEATURES

print(f"\n── Mức độ quan trọng các đặc trưng ──")
for feat, imp in sorted(zip(feat_names, importances), key=lambda x: -x[1]):
    bar = "█" * int(imp * 60)
    print(f"   {feat:<28} : {imp:.4f}  {bar}")

# Đánh giá mô hình + Hình ảnh

In [ ]:
y_pred       = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

In [ ]:
acc  = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_pred_proba)
cm   = confusion_matrix(y_test, y_pred)
cr   = classification_report(
    y_test, y_pred,
    target_names=le_target.classes_
)

In [ ]:
print("=" * 58)
print(f"  ✅ Accuracy (Độ chính xác) : {acc * 100:.2f}%")
print(f"  📉 Log Loss (Hàm lỗi)     : {loss:.6f}")
print("=" * 58)

print("\n── Confusion Matrix ──")
cm_df = pd.DataFrame(
    cm,
    index  =[f"Thực: {c}"    for c in le_target.classes_],
    columns=[f"Dự đoán: {c}" for c in le_target.classes_]
)
print(cm_df.to_string())

print("\n── Classification Report ──")
print(cr)

print(f"Giải thích Log Loss = {loss:.4f}")
print("→ Giá trị càng gần 0 → mô hình dự đoán xác suất càng tốt.")

In [ ]:
# --- 6.3 Hình ảnh tổng hợp kết quả
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a", "axes.facecolor": "#1a1a2e",
    "axes.edgecolor": "#444466",   "axes.labelcolor": "#ccccee",
    "axes.titlecolor": "#ffffff",  "xtick.color": "#aaaacc",
    "ytick.color": "#aaaacc",      "text.color": "#ccccee",
    "grid.color": "#2a2a4a",       "grid.linewidth": 0.6,
    "font.family": "monospace",
})
PALETTE = ["#7b68ee", "#00d4aa", "#ff6b8a"]

fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor("#0f0f1a")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.38)

# ── Header ──────────────────────────────────────────────────
ax_h = fig.add_subplot(gs[0, :])
ax_h.axis("off")
ax_h.text(0.5, 0.65,
    "KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH — RANDOM FOREST  |  Shopee TH Dataset",
    ha="center", fontsize=14, fontweight="bold", color="#ffffff")
ax_h.text(0.5, 0.1,
    f"Accuracy: {acc*100:.2f}%   |   Log Loss: {loss:.6f}   |   "
    f"Train: {X_train.shape[0]:,} mẫu   |   Test: {X_test.shape[0]:,} mẫu   |   "
    f"Features: {X.shape[1]}   |   Trees: 200",
    ha="center", fontsize=10, color="#aaddff")

# ── Subplot 1: Confusion Matrix ──────────────────────────────
ax1 = fig.add_subplot(gs[1, 0])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le_target.classes_
)
disp.plot(ax=ax1, colorbar=False, cmap="Blues")
ax1.set_title("Confusion Matrix", fontsize=11, fontweight="bold", pad=10)
ax1.set_facecolor("#1a1a2e")
for text in disp.text_.ravel():
    text.set_color("white")
    text.set_fontsize(11)
    text.set_fontweight("bold")

# ── Subplot 2: Feature Importance ────────────────────────────
ax2 = fig.add_subplot(gs[1, 1])
ax2.set_facecolor("#1a1a2e")
sorted_idx = np.argsort(importances)
feat_labels = [ALL_FEATURES[i].replace("_", " ") for i in sorted_idx]
bars = ax2.barh(feat_labels, importances[sorted_idx],
                color=[PALETTE[i % 3] for i in range(len(sorted_idx))],
                edgecolor="#333355", linewidth=0.4, height=0.6)
for bar, val in zip(bars, importances[sorted_idx]):
    ax2.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
             f"{val:.3f}", va="center", fontsize=8, color="white")
ax2.set_title("Feature Importance", fontsize=11, fontweight="bold", pad=10)
ax2.set_xlabel("Importance Score")
ax2.grid(True, axis="x", alpha=0.25)
ax2.set_xlim(0, importances.max() + 0.06)
for sp in ax2.spines.values():
    sp.set_edgecolor("#333355")

# ── Subplot 3: Precision / Recall / F1 per class ─────────────
ax3 = fig.add_subplot(gs[1, 2])
ax3.set_facecolor("#1a1a2e")

from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, labels=[0, 1, 2]
)
classes   = list(le_target.classes_)
x         = np.arange(len(classes))
width     = 0.25

ax3.bar(x - width, prec, width, label="Precision", color=PALETTE[0], alpha=0.85, edgecolor="#333355")
ax3.bar(x,         rec,  width, label="Recall",    color=PALETTE[1], alpha=0.85, edgecolor="#333355")
ax3.bar(x + width, f1,   width, label="F1-Score",  color=PALETTE[2], alpha=0.85, edgecolor="#333355")

for i, (p, r, f) in enumerate(zip(prec, rec, f1)):
    ax3.text(i - width, p + 0.01, f"{p:.2f}", ha="center", fontsize=8, color="white")
    ax3.text(i,         r + 0.01, f"{r:.2f}", ha="center", fontsize=8, color="white")
    ax3.text(i + width, f + 0.01, f"{f:.2f}", ha="center", fontsize=8, color="white")

ax3.set_xticks(x)
ax3.set_xticklabels(classes, fontsize=10)
ax3.set_ylim(0, 1.15)
ax3.set_title("Precision / Recall / F1 theo lớp", fontsize=11, fontweight="bold", pad=10)
ax3.set_ylabel("Score")
ax3.legend(framealpha=0.15, labelcolor="white", fontsize=9)
ax3.grid(True, axis="y", alpha=0.25)
for sp in ax3.spines.values():
    sp.set_edgecolor("#333355")

plt.savefig("phan_4_ket_qua_mo_hinh.png", dpi=150,
            bbox_inches="tight", facecolor="#0f0f1a")
plt.show()
print("✅ Đã lưu: phan_4_ket_qua_mo_hinh.png")